In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import nltk
from nltk.tokenize import sent_tokenize

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification)

from tqdm.auto import tqdm
import json

In [2]:
#REVIEWS_FILE="/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_reviews_fantasy_paranormal.json"
#BOOKS_FILE = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_books_fantasy_paranormal.json"

REVIEWS_FILE = "/scratch/kk01697/data/raw/goodreads/goodreads_reviews_fantasy_paranormal.json"
BOOKS_FILE = "/scratch/kk01697/data/raw/goodreads/goodreads_books_fantasy_paranormal.json"

MODEL_DIR="/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/scripts/objective1/outputs/deberta_classifier/checkpoint-544"

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
custom_dim=['Narrative Structure & Quality','Character & Emotion','Originality','Immersion','Thematic Depth','Writing Style']


In [3]:
books = []
with open(BOOKS_FILE, "r") as f:
    for line in tqdm(f):
        books.append(json.loads(line))
book_df = pd.DataFrame(books)
print(book_df.shape)
book_df.head()

reviews = []
with open(REVIEWS_FILE, "r") as f:
    for line in tqdm(f):
        reviews.append(json.loads(line))
reviews_df = pd.DataFrame(reviews)
print(reviews_df.shape)
reviews_df.head()

0it [00:00, ?it/s]

(258585, 29)


0it [00:00, ?it/s]

(3424641, 11)


,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,18245960,dfdbb7b0eb5a7e4c26d59a937e2e5feb,5,This is a special book. It started slow for ab...,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,Sat Aug 26 12:05:52 -0700 2017,Tue Aug 15 13:23:18 -0700 2017,28,1
1,8842281e1d1347389f2ab93d60773d4d,5577844,52c8ac49496c153e4a97161e36b2db55,5,A beautiful story. Neil Gaiman is truly a uniq...,Wed Sep 24 09:29:29 -0700 2014,Wed Oct 01 00:31:56 -0700 2014,Tue Sep 30 00:00:00 -0700 2014,Sun Sep 21 00:00:00 -0700 2014,5,1
2,8842281e1d1347389f2ab93d60773d4d,17315048,885c772fb033b041f42d57cef5be0a43,5,Mark Watney is a steely-eyed missile man. A ma...,Sat Apr 05 09:30:53 -0700 2014,Wed Mar 22 11:33:10 -0700 2017,Mon Aug 25 00:00:00 -0700 2014,Sat Aug 16 00:00:00 -0700 2014,25,5
3,8842281e1d1347389f2ab93d60773d4d,13453029,46a6e1a14e8afc82d221fec0a2bd3dd0,4,A fun fast paced book that sucks you in right ...,Tue Dec 04 11:12:22 -0800 2012,Sat Jul 26 11:43:28 -0700 2014,Tue Jul 08 00:00:00 -0700 2014,Wed Jul 02 00:00:00 -0700 2014,5,1
4,8842281e1d1347389f2ab93d60773d4d,13239822,a582bfa8efd69d453a5a21a678046b36,3,"This book has a great premise, and is full of ...",Mon Jul 02 16:04:16 -0700 2012,Wed Mar 22 11:32:20 -0700 2017,Wed Aug 15 00:00:00 -0700 2012,Sun Aug 12 00:00:00 -0700 2012,7,0


In [4]:
merged = reviews_df.merge(book_df, on='book_id', how='inner')

In [5]:
merged["review_id"].nunique()

3424641

In [6]:
def basic_filter(df: pd.DataFrame,min_words: int = 30, min_rating: int = 1, max_rating: int = 5) -> pd.DataFrame:
    """ Remove empty / very short reviews and invalid ratings. """
    english_codes = ["eng", "en-UK", "en-US", "en-AUS"]

    df = df[df["language_code"].isin(english_codes)]
    df = df.dropna(subset=["review_text", "rating"])
    df = df[df["review_text"].str.split().str.len() >= min_words]
    df = df[df["rating"].between(min_rating, max_rating)]
    df = df.drop_duplicates(subset=["review_text"])
    df = df.reset_index(drop=True)
    return df

df_raw = merged
df = basic_filter(df_raw)
print(f"Loaded {len(df_raw):,} rows → {len(df):,} after filtering")

Loaded 3,424,641 rows → 1,903,442 after filtering


In [17]:
## not working with all 1,9m reviews
filter_sample_ids = df["review_id"].drop_duplicates().sample(n=50000, random_state=42)
filter_df = df[df["review_id"].isin(filter_sample_ids)]
print(filter_df.shape)
filter_df.to_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/filtered_reviews.csv")

(50000, 39)


In [18]:
filter_df=filter_df[['review_id','review_text']]
filter_df=filter_df.dropna()
print(len(filter_df))

50000


In [19]:
def tokenise_reviews(df):
    rows = []

    for _, r in tqdm(df.iterrows(), total=len(df), desc="Tokenising reviews"):
        sents = sent_tokenize(r.review_text)

        for i, s in enumerate(sents):
            rows.append({"review_id": r.review_id, "sentence_idx": i, "sentence": s,})

    return pd.DataFrame(rows)

tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model=model.to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerN

In [10]:
def predict_sentences(sentences: list[str], tokenizer,model, threshold: float = 0.5, batch_size: int = 64,) -> np.ndarray:
    all_preds = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="Predicting", unit="batch",):
        batch = sentences[i : i + batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**enc).logits

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= threshold).astype(int)
        all_preds.append(preds)

    return np.vstack(all_preds)

In [20]:
sentence_df=tokenise_reviews(filter_df)
##preds=predict_sentences(sentence_df.sentence.tolist())
preds = predict_sentences(sentence_df["sentence"].tolist(),tokenizer,model)
pred_df=pd.DataFrame(preds,columns=custom_dim)
sentence_predictions=pd.concat([sentence_df, pred_df],axis=1)

Tokenising reviews:   0%|          | 0/50000 [00:00<?, ?it/s]

Predicting:   0%|          | 0/9276 [00:00<?, ?batch/s]

In [23]:
review_scores=(sentence_predictions.groupby('review_id')[custom_dim].mean().reset_index())

sentence_predictions.to_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/scripts/data/sentence_predictions.csv", index=False)
review_scores.to_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/scripts/data/review_scores.csv", index=False)

In [22]:
review_scores.head()

,review_id,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,0000ccecd0a6252191c95c29b27f5a82,0.0,0.0,0.0,0.0,0.0,0.0
1,0000fbf05c6d154f1ae4362b09e02867,0.0,0.0,0.0,0.0,0.0,0.0
2,000316a11e1b9e00becc4430ffd320d5,0.0,0.0,0.0,0.0,0.0,0.0
3,0004140724746684fe336a8d7cfd950b,0.0,0.0,0.0,0.0,0.0,0.0
4,0004294ef8c613dd1bc2a39d3592f4ae,0.0,0.0,0.0,0.0,0.0,0.0
